# Breast Cancer Subtype Prediction Project
# Multi-Omics Integration (RNA-Seq + Methylation)
# Step 1: Reading & Data Synchronization

## Import Library

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
from src.utils import ensure_directories, save_object

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 20)

print('✅ Libraries & Utilities Imported Successfully.')

✅ Libraries & Utilities Imported Successfully.


## Define Functions

In [2]:
# Output directory management and saving delegated to src.utils
ensure_directories('../outputs')
print('✅ Directory structure verified.')

✅ Directory structure verified.


## Load Data & Set Paths

In [3]:
data_dir = '../data'

clinical_file = os.path.join(data_dir, 'Human_TCGA_BRCA_MS_Clinical_Clinical_01_28_2016_BI_Clinical_Firehose.tsi')
rna_file = os.path.join(data_dir, 'Human_TCGA_BRCA_UNC_RNAseq_HiSeq_RNA_01_28_2016_BI_Gene_Firehose.gz')
meth_file = os.path.join(data_dir, 'Human_TCGA_BRCA_JHU_USC_Methylation_Meth450_01_28_2016_BI_Gene_Firehose.gz')

print(f'Checking for data in: {data_dir}')

for p in [clinical_file, rna_file, meth_file]:
    if not os.path.exists(p):
        raise FileNotFoundError(f'❌ Required file missing: {p}')
print('✅ All raw data files located.')

Checking for data in: ../data
✅ All raw data files located.


## Process Clinical Data (Extract PAM50 Labels)

In [4]:
print('Loading Clinical Data...')

clinical = pd.read_csv(clinical_file, sep='\t', index_col=0).T
if 'PAM50' not in clinical.columns:
    raise ValueError("Error: Column 'PAM50' Not Found in Clinical Data!")

# Fix SettingWithCopy by using explicit .copy()
clinical_clean = clinical.dropna(subset=['PAM50']).copy()

label_mapping = {'LumA': 0, 'LumB': 1, 'Her2': 2, 'Basal': 3}
clinical_clean = clinical_clean[clinical_clean['PAM50'].isin(label_mapping.keys())].copy()
clinical_clean['label'] = clinical_clean['PAM50'].map(label_mapping).astype(int)

# Dynamically derive target_names from label_mapping for present classes
target_names = [k for k, v in sorted(label_mapping.items(), key=lambda x: x[1]) if v in np.unique(clinical_clean['label'])]

print(f'✅ Clinical Data Processed. Valid Patients: {clinical_clean.shape[0]}')
print('Target Names:', target_names)
print('Subtype Distribution:\n', clinical_clean['PAM50'].value_counts())

Loading Clinical Data...
✅ Clinical Data Processed. Valid Patients: 826
Target Names: ['LumA', 'LumB', 'Her2', 'Basal']
Subtype Distribution:
 PAM50
LumA     426
LumB     186
Basal    147
Her2      67
Name: count, dtype: int64


## Load Omics Data (RNA-Seq & Methylation)

In [5]:
print("Loading Omics Data (This may take a moment)...")

rna = pd.read_csv(rna_file, sep='\t', index_col=0).T
print(f"✅ RNA-Seq Loaded. Shape: {rna.shape}")

meth = pd.read_csv(meth_file, sep='\t', index_col=0).T
print(f"✅ Methylation Loaded. Shape: {meth.shape}")

Loading Omics Data (This may take a moment)...


✅ RNA-Seq Loaded. Shape: (1093, 20155)


✅ Methylation Loaded. Shape: (783, 20106)


## Processing and Matching Samples

In [6]:
print('Synchronizing patients across all datasets...')

common_patients = clinical_clean.index.intersection(rna.index).intersection(meth.index)

# Assertions against duplicate sample IDs
assert not rna.index.duplicated().any(), 'Duplicate sample IDs in RNA'
assert not meth.index.duplicated().any(), 'Duplicate sample IDs in Methylation'
assert not clinical_clean.index.duplicated().any(), 'Duplicate sample IDs in Clinical'

print('------------------------------------------------')
print(f'Patients in Clinical: {clinical_clean.shape[0]}')
print(f'Patients in RNA-Seq:  {rna.shape[0]}')
print(f'Patients in Methylation: {meth.shape[0]}')
print('------------------------------------------------')
print(f'🚀 FINAL COMMON PATIENTS: {len(common_patients)}')

if len(common_patients) == 0:
    raise RuntimeError('❌ Critical Error: No common patients found. Check Patient IDs.')

Synchronizing patients across all datasets...
------------------------------------------------
Patients in Clinical: 826
Patients in RNA-Seq:  1093
Patients in Methylation: 783
------------------------------------------------
🚀 FINAL COMMON PATIENTS: 549


## Saving Result

In [7]:
print("Saving processed datasets to '../outputs/'...")

X_rna_final = rna.loc[common_patients].astype(np.float32)
X_meth_final = meth.loc[common_patients].astype(np.float32)
y_final = clinical_clean.loc[common_patients, 'label']

save_object(X_rna_final, 'X_rna_raw')
save_object(X_meth_final, 'X_meth_raw')
save_object(y_final, 'y_labels')
save_object(common_patients, 'patient_ids')
save_object(list(X_rna_final.columns), 'rna_feature_names')
save_object(list(X_meth_final.columns), 'meth_feature_names')

print('\n🎉 Reading Step Completed Successfully.')

Saving processed datasets to '../outputs/'...


💾 Saved: ../outputs\X_rna_raw.parquet (+ .pkl)


💾 Saved: ../outputs\X_meth_raw.parquet (+ .pkl)
💾 Saved: ../outputs\y_labels.parquet (+ .pkl)
💾 Saved: ../outputs\patient_ids.pkl
💾 Saved: ../outputs\rna_feature_names.pkl
💾 Saved: ../outputs\meth_feature_names.pkl

🎉 Reading Step Completed Successfully.
